> Copyright 2026 Google LLC.
>
> Licensed under the Apache License, Version 2.0 (the "License");
> you may not use this file except in compliance with the License.
> You may obtain a copy of the License at
>
>      http://www.apache.org/licenses/LICENSE-2.0
>
> Unless required by applicable law or agreed to in writing, software
> distributed under the License is distributed on an "AS-IS" BASIS,
> WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
> See the License for the specific language governing permissions and
> limitations under the License.

# Demo of IBTrACS gridding and running the direct tracker

This colab is a demo for the IBTrACS gridding process and running the direct
tracker on the gridded data. First, it runs the gridding process on a small
subset of IBTrACS data (5 days in 2024) and visualizes the intermediate
outputs (e.g. cyclone existence field). It then runs the direct tracker on
these ground truth data and plots the tracked blobs on top of the gridded
data. Finally, it checks that the tracker output matches the ground truth
up to a specified tolerance.

## Set up

In [ ]:
# @title Pip install repo and dependencies and reconfigure jax if running on TPU.
%pip install --upgrade https://github.com/google-deepmind/weathernext/archive/master.zip

In [ ]:
# @title Imports

# Download the IBTrACS dataset for the last 3 years, as made available by
# NOAA.
!wget https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/netcdf/IBTrACS.last3years.v04r01.nc
_IBTRACS_TABULAR_DATA_NC_PATH = "IBTrACS.last3years.v04r01.nc"
_NETCDF_ENGINE = "h5netcdf"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython import display
import xarray as xr
from scipy import optimize
import tqdm

# Suppress an expected warning that occurs during quadrant imputation in
# ibtracs_processing_utils.py.
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")

from weathernext.cyclones import constants
from weathernext.cyclones import cyclone_utils
from weathernext.cyclones import direct_tracker_6h_v1_config
from weathernext.cyclones import ibtracs_processing_stages
from weathernext.cyclones import ibtracs_processing_utils
from weathernext.cyclones import ibtracs_netcdf_to_csv
from weathernext.cyclones import tracker_utils


In [ ]:
# @title 0. Define constants for gridding
_EXISTENCE_GAUSSIAN_AND_DISC_RADIUS_KM = 120.0
_SCALAR_VARIABLE_DISC_RADIUS_KM = 240.0
_QUADRANT_NAN_MASK_DISC_RADIUS_KM = 800.0
_QUADRANT_NUM_SUBGRID = 8
_CORE_CYCLONE_VARIABLES_ONLY = True

START_DATE = "2024-10-07"
TEST_YEAR = int(START_DATE[:4])
NUM_DAYS = 5
RESOLUTION = 1.0

_WIND_CALIBRATION_START_DATE = "2023-01-01"
_WIND_CALIBRATION_END_DATE = "2025-12-31"


In [ ]:
# @title 1. Load data and convert to CSV
print(f"Loading raw IBTrACS NetCDF from: {_IBTRACS_TABULAR_DATA_NC_PATH}")


def add_start_and_end_time_coords(ds: xr.Dataset) -> xr.Dataset:
  start_time = ds.time.min("date_time", skipna=True)
  end_time = ds.time.max("date_time", skipna=True)
  return ds.assign_coords({"start_time": start_time, "end_time": end_time})


def project_lon_to_180_range(lon: np.ndarray) -> np.ndarray:
  return ((lon + 180) % 360) - 180


ibtracs_data = xr.open_dataset(
    _IBTRACS_TABULAR_DATA_NC_PATH,
    engine=_NETCDF_ENGINE,
).compute()

ibtracs_data = add_start_and_end_time_coords(ibtracs_data)
ibtracs_data["time"] = ibtracs_data["time"].dt.round("1s")
ibtracs_data[constants.LON] = project_lon_to_180_range(
    ibtracs_data[constants.LON]
)
all_storms_df, initial_storms_df = (
    ibtracs_netcdf_to_csv.prepare_ibtracs_storms_dfs(
        ibtracs_ds=ibtracs_data,
        init_time=np.datetime64(START_DATE),
    )
)



In [ ]:
# @title 2. Grid IBTrACS
# NOTE: this step takes about 5min
def _unpack_variables(packed_dataset: xr.Dataset) -> xr.Dataset:
  labels_to_unpack = set([
      "single_level_temporal",
      "multilevel_temporal",
      "single_level_static",
      "multi_model_level_temporal",
  ])
  labels_available = set(packed_dataset.keys())
  labels_to_unpack = labels_to_unpack.intersection(labels_available)
  other_labels = labels_available - labels_to_unpack
  return xr.merge(
      [packed_dataset[list(other_labels)]]
      + [
          packed_dataset[label].to_dataset(dim=f"{label}_variable")
          for label in list(labels_to_unpack)
      ]
  )


def _grid_ibtracs_for_date_range(
    ibtracs_data, start_date, num_days, resolution, wind_calibration_model
):
  t0 = np.datetime64(start_date)
  times = np.arange(
      t0, t0 + np.timedelta64(num_days, "D"), np.timedelta64(6, "h")
  )
  gridded_frames = []
  for i, time in enumerate(tqdm.tqdm(times, desc="Gridding IBTrACS")):
    ibtracs_slice = ibtracs_processing_stages.load_time_slice_from_ibtracs(
        ibtracs_data=ibtracs_data, datetime=time, time_index=i
    )
    gridded = ibtracs_processing_stages.convert_ibtracs_to_gridded_data(
        ibtracs_data=ibtracs_slice,
        resolution=resolution,
        existence_gaussian_and_disc_radius_km=_EXISTENCE_GAUSSIAN_AND_DISC_RADIUS_KM,
        scalar_variable_disc_radius_km=_SCALAR_VARIABLE_DISC_RADIUS_KM,
        quadrant_nan_mask_disc_radius_km=_QUADRANT_NAN_MASK_DISC_RADIUS_KM,
        quadrant_num_subgrid=_QUADRANT_NUM_SUBGRID,
        core_cyclone_variables_only=_CORE_CYCLONE_VARIABLES_ONLY,
        wind_calibration_model=wind_calibration_model,
    )
    gridded_frames.append(_unpack_variables(gridded.grid_data).isel(time=0))
  return xr.concat(gridded_frames, dim="time")


def _prepare_gridded_data_for_tracker(ds):
  rename_map = {}
  if "latitude" in ds.dims:
    rename_map["latitude"] = "lat"
  if "longitude" in ds.dims:
    rename_map["longitude"] = "lon"
  if rename_map:
    ds = ds.rename(rename_map)
  init_time = pd.Timestamp(ds.time.values[0])
  lead_times = ds.time.values - ds.time.values[0]
  ds = ds.assign_coords(time=lead_times, init_time=init_time)
  return ds


wind_calibration_model = (
    ibtracs_processing_utils.create_wind_speed_calibration_and_aggregation_model(
        ibtracs_data=ibtracs_data,
        wind_calibration_start_date=_WIND_CALIBRATION_START_DATE,
        wind_calibration_end_date=_WIND_CALIBRATION_END_DATE,
        wind_calibration_skip_if_no_overlap=True,
    )
)

gridded_ds_raw = _grid_ibtracs_for_date_range(
    ibtracs_data, START_DATE, NUM_DAYS, RESOLUTION, wind_calibration_model
)
gridded_ds = _prepare_gridded_data_for_tracker(gridded_ds_raw)

# Check that the calibrated wind variable was created during gridding.
all_wind_disc_var = f"cyclone_{ibtracs_processing_utils.ALL_WIND_VARIABLE}_disc"
assert all_wind_disc_var in gridded_ds.data_vars, (
    f"Gridded dataset does not contain '{all_wind_disc_var}' "
    f"variable: {list(gridded_ds.data_vars)}"
)
# We do not need the all wind disc variable in the round trip test because it
# does not exist in the ground truth data in the first place. We also don't
# track this variable with the direct tracker (it is only used in training).
gridded_ds = gridded_ds.drop_vars(all_wind_disc_var)


In [ ]:
# Visualise Probability Fields (Existence)
exists_var = "cyclone_exists_gaussian_unit_mode"
if exists_var not in gridded_ds:
  print(
      f"{exists_var} not found. Available variables: {list(gridded_ds.keys())}"
  )
else:
  fig, ax = plt.subplots(figsize=(10, 5))
  frame = gridded_ds[exists_var].isel(time=0)
  im = frame.plot.imshow(ax=ax, cmap="viridis", vmin=0, vmax=1)

  def update_existence(lead_time):
    curr_frame = gridded_ds[exists_var].sel(time=lead_time)
    vt = pd.Timestamp(gridded_ds.init_time.values) + lead_time
    im.set_array(curr_frame.values)
    ax.set_title(f"Cyclone Existence Field: {exists_var}\nValid Time: {vt}")
    return (im,)

  anim = animation.FuncAnimation(
      fig, update_existence, frames=gridded_ds.time.values, blit=False
  )
  plt.close(fig)
  # display() is available in IPython environment
  display.display(display.HTML(anim.to_jshtml()))


In [ ]:
# @title 3. Create initial storms and run Direct Tracker

config = direct_tracker_6h_v1_config.get_config()
config.tracker_kwargs["cyclogenesis_minimum_duration"] = None
# NOTE: for this round trip demo, we do not apply the physical consistency
# post-processing step, since we want to check that the tracker output is
# identical to the ground truth file, but include the consistency implementation
# in the source code to demonstrate how it works.
config.tracker_kwargs["enforce_physically_consistent_quadrants_and_winds"] = (
    False
)
tracker = config.tracker_constructor(**config.tracker_kwargs)

predicted_tracks = tracker(
    gridded_ds=gridded_ds,
    initial_storms_df=initial_storms_df,
    do_cyclogenesis=True,
)
predicted_tracks[constants.LON] = predicted_tracks[constants.LON] % 360


In [ ]:
# @title 4. Visualise output of tracker overlaid on probability fields
if exists_var in gridded_ds:
  fig, ax = plt.subplots(figsize=(10, 5))
  frame = gridded_ds[exists_var].isel(time=0)
  im = frame.plot.imshow(ax=ax, cmap="viridis", vmin=0, vmax=1)
  sc = ax.scatter([], [], color="red", marker="x", s=50, label="Tracked Centers")
  ax.legend()

  def update_tracker(lead_time):
    curr_frame = gridded_ds[exists_var].sel(time=lead_time)
    vt = pd.Timestamp(gridded_ds.init_time.values) + lead_time
    im.set_array(curr_frame.values)

    preds_vt = predicted_tracks[predicted_tracks[constants.VALID_TIME] == vt]
    if not preds_vt.empty:
      sc.set_offsets(preds_vt[[constants.LON, constants.LAT]].values)
    else:
      # set_offsets expects a 2D array
      sc.set_offsets(np.zeros((0, 2)))

    ax.set_title(f"Tracker Output & Existence Field\nValid Time: {vt}")
    return (im, sc)

  anim = animation.FuncAnimation(
      fig, update_tracker, frames=gridded_ds.time.values, blit=False
  )
  plt.close(fig)
  display.display(display.HTML(anim.to_jshtml()))


In [ ]:
# @title 5. Round trip numerical assertions
_MAX_MATCH_DISTANCE_KM = 20.0
_SCALAR_COLUMNS_TO_CHECK = [
    constants.MIN_SEA_LEVEL_PRESSURE_HPA,
    constants.MAX_SUSTAINED_WIND_SPEED_KNOTS,
    constants.RADIUS_OF_MAXIMUM_WINDS,
] + list(constants.QUADRANT_RADII_FLATTENED)


def _match_rows_by_geodesic_distance(gt_df, pred_df):
  gt_latlons = gt_df[[constants.LAT, constants.LON]].values
  pred_latlons = pred_df[[constants.LAT, constants.LON]].values
  cost_matrix = cyclone_utils.geodesic_distance(
      gt_latlons[:, np.newaxis, :],
      pred_latlons[np.newaxis, :, :],
  )
  gt_indices, pred_indices = optimize.linear_sum_assignment(cost_matrix)
  return gt_indices, pred_indices, cost_matrix


def assert_tracker_roundtrip(
    all_storms_df: pd.DataFrame,
    predicted_tracks: pd.DataFrame,
    start_date: str,
    num_days: int,
    max_match_distance_km: float = _MAX_MATCH_DISTANCE_KM,
    scalar_columns_to_check: list[str] = _SCALAR_COLUMNS_TO_CHECK,
    strict: bool = True,
):
  """Asserts tracker output matches ground truth within tolerance."""
  t1 = pd.Timestamp(start_date) + pd.Timedelta(days=num_days)
  mask = (all_storms_df[constants.VALID_TIME] >= pd.Timestamp(start_date)) & (
      all_storms_df[constants.VALID_TIME] < t1
  )
  ground_truth_df = all_storms_df[mask].copy()
  ground_truth_df[constants.LON] = ground_truth_df[constants.LON] % 360

  gt_valid_times = sorted(ground_truth_df[constants.VALID_TIME].unique())
  pred_valid_times = sorted(predicted_tracks[constants.VALID_TIME].unique())

  def _handle_error(msg, raise_error=strict, gt_row=None, pred_row=None):
    if raise_error:
      raise AssertionError(msg)
    else:
      print(120 * "=")
      print(f"WARNING: {msg}")
      print(120 * "=")
      if gt_row is not None and pred_row is not None:
        expected = gt_row.to_frame()
        predicted = pred_row.to_frame().loc[expected.index, :]
        comparison_df = expected.compare(
            predicted, result_names=('ibtracs csv', 'track from gridded'),
            keep_equal=True,
        )
        comparison_df.columns = comparison_df.columns.get_level_values(1)
        display.display(comparison_df)

  if gt_valid_times != pred_valid_times:
    _handle_error(
        f"Unique valid times differ! GT: {gt_valid_times}, Pred:"
        f" {pred_valid_times}"
    )

  for vt in gt_valid_times:
    gt_at_vt = ground_truth_df[
        ground_truth_df[constants.VALID_TIME] == vt
    ].reset_index(drop=True)
    pred_at_vt = predicted_tracks[
        predicted_tracks[constants.VALID_TIME] == vt
    ].reset_index(drop=True)

    if len(gt_at_vt) != len(pred_at_vt):
      print(f"GT rows: {gt_at_vt}")
      print(f"Pred rows: {pred_at_vt}")
      _handle_error(
          f"Row count mismatch at vt={vt}. GT: {len(gt_at_vt)}, Pred:"
          f" {len(pred_at_vt)}"
      )

    if len(gt_at_vt) == 0:
      continue

    gt_indices, pred_indices, cost_matrix = _match_rows_by_geodesic_distance(
        gt_at_vt, pred_at_vt
    )

    for gi, pi in zip(gt_indices, pred_indices):
      gt_row = gt_at_vt.iloc[gi]
      pred_row = pred_at_vt.iloc[pi]
      dist_km = cost_matrix[gi, pi]

      if dist_km >= max_match_distance_km:
        _handle_error(
            f"Matched rows {dist_km:.2f} km apart (limit:"
            f" {max_match_distance_km}) at {vt}",
            gt_row=gt_row,
            pred_row=pred_row
        )

      for col in scalar_columns_to_check:
        gt_val = gt_row[col]
        pred_val = pred_row[col]
        if pd.isna(gt_val):
          if col in constants.QUADRANT_RADII_FLATTENED:
            if not (pd.isna(pred_val) or float(pred_val) == 0.0):
              _handle_error(
                  f"Got {pred_val} at {vt} for '{col}' when gt is NaN. This can"
                  " happen for quadrant radii due to the mean-imputation that"
                  " is applied to the ground truth data during gridding, or due"
                  " to the physical consistency enforcement step (if enabled).",
                  gt_row=gt_row,
                  pred_row=pred_row,
                  raise_error=False,
              )
          else:
            if not pd.isna(pred_val):
              _handle_error(
                  f"Expected NaN for '{col}' when gt is NaN. Got {pred_val} at"
                  f" {vt}",
                  gt_row=gt_row,
                  pred_row=pred_row
              )
          continue
        if not np.isclose(float(gt_val), float(pred_val), atol=1e-3, rtol=1e-3):
          _handle_error(
              f"Value mismatch for '{col}': gt={gt_val}, pred={pred_val} at {vt}",
              gt_row=gt_row,
              pred_row=pred_row
          )


assert_tracker_roundtrip(
    all_storms_df=all_storms_df,
    predicted_tracks=predicted_tracks,
    start_date=START_DATE,
    num_days=NUM_DAYS,
)

print(
    "All round trip numerical assertions passed successfully: this means that\n"
    "the tracks from applying the direct tracker on gridded IBTrACS are\n"
    "within the specified tolerance of the ground truth CSV tracks."
)


In [ ]:
# @title 6. Enforcing physical consistency in predictions
# This cell will log any inconsistencies in the round trip caused by enforcing
# physically consistent quadrants and winds. There are some cases in the ground
# truth data where, for example, the max wind speed is over the speed threshold
# of a given quadrant but the quadrant radii themselves are all-nan (undefined /
# missing). These cases result in a non-exact match in the round trip when we
# enforce physical consistency, so we log all these cases without raising an
# error here.
print("Enforcing physical consistency on predicted tracks...")
predicted_tracks_consistent = tracker_utils.enforce_physical_consistency_on_quadrants_and_winds(
    predicted_tracks
)

print("\nRunning round trip verification with strict=False:")
assert_tracker_roundtrip(
    all_storms_df=all_storms_df,
    predicted_tracks=predicted_tracks_consistent,
    start_date=START_DATE,
    num_days=NUM_DAYS,
    strict=False,
)
